### Pacotes importados

In [1]:
# Imports
using LinearAlgebra
using Printf

## Chapter 8: Quasi-Newton methods

### Algorithm 8.1: Finite difference Newton's method: one variable

![image.png](attachment:a6f28177-acdd-49e6-b027-d5a646210591.png)

Example: $F(x)=x^2-2$. Run the example with $x_0=2$ and $\tau=10^{-7}$

In [2]:
# Algorithm 8.1: Finite difference Newton's method - one variable

function finite_difference_newton_1d(F, x0; τ=1e-7, h=1e-6, max_iter=100)
    x = float(x0)

    println("Finite Difference Newton Method - 1 variable")
    println("Initial x = ", x)
    println("Tolerance = ", τ)
    println()

    for k in 0:max_iter
        Fx = F(x)

        @printf("iter = %2d | x = %.10f | F(x) = %.10e\n", k, x, Fx)

        if abs(Fx) <= τ
            println("\nConverged.")
            return x
        end

        # finite difference derivative
        dF = (F(x + h) - F(x)) / h

        if abs(dF) < 1e-12
            error("Derivative too close to zero.")
        end

        # Newton update
        x = x - Fx / dF
    end

    println("\nMaximum number of iterations reached.")
    return x
end

F(x) = x^2 - 2

x0 = 2.0
τ = 1e-7

x_star = finite_difference_newton_1d(F, x0; τ=τ)
println("\nApproximate solution: x* = ", x_star)
println("F(x*) = ", F(x_star))

Finite Difference Newton Method - 1 variable
Initial x = 2.0
Tolerance = 1.0e-7

iter =  0 | x = 2.0000000000 | F(x) = 2.0000000000e+00
iter =  1 | x = 1.5000001251 | F(x) = 2.5000037524e-01
iter =  2 | x = 1.4166667014 | F(x) = 6.9445427960e-03
iter =  3 | x = 1.4142156872 | F(x) = 6.0099212624e-06
iter =  4 | x = 1.4142135624 | F(x) = 6.6400218657e-12

Converged.

Approximate solution: x* = 1.4142135623754426
F(x*) = 6.640021865678136e-12


Run the example with $x_0=2$ and $\tau=0.1$

In [3]:
# Same example with tolerance τ = 0.1

F(x) = x^2 - 2

x0 = 2.0
τ = 0.1

x_star = finite_difference_newton_1d(F, x0; τ=τ)
println("\nApproximate solution: x* = ", x_star)
println("F(x*) = ", F(x_star))

Finite Difference Newton Method - 1 variable
Initial x = 2.0
Tolerance = 0.1

iter =  0 | x = 2.0000000000 | F(x) = 2.0000000000e+00
iter =  1 | x = 1.5000001251 | F(x) = 2.5000037524e-01
iter =  2 | x = 1.4166667014 | F(x) = 6.9445427960e-03

Converged.

Approximate solution: x* = 1.4166667013789846
F(x*) = 0.006944542796013309


![image.png](attachment:e520e15e-4279-432d-9659-5191b6e24a8b.png)

Example: $F(x)=x^2-2$, with $x_0=2$ and $a_0=1$

In [4]:
# Algorithm 8.2: Secant method - one variable
# Example: F(x) = x^2 - 2, x0 = 2 and a0 = 1

function secant_method_1d(F, x0, a0; τ=1e-7, max_iter=100)
    x_prev = float(x0)
    x = float(x0 + a0)

    println("Secant Method - 1 variable")
    println("x0 = ", x_prev, ", x1 = ", x)
    println("Tolerance = ", τ)
    println()

    for k in 1:max_iter
        Fx = F(x)
        Fprev = F(x_prev)

        @printf("iter = %2d | x = %.10f | F(x) = %.10e\n", k, x, Fx)

        if abs(Fx) <= τ
            println("\nConverged.")
            return x
        end

        denom = Fx - Fprev
        if abs(denom) < 1e-12
            error("Secant denominator too close to zero.")
        end

        x_new = x - Fx * (x - x_prev) / denom
        x_prev = x
        x = x_new
    end

    println("\nMaximum number of iterations reached.")
    return x
end

F(x) = x^2 - 2

x0 = 2.0
a0 = 1.0
τ = 1e-7

x_star = secant_method_1d(F, x0, a0; τ=τ)
println("\nApproximate solution: x* = ", x_star)
println("F(x*) = ", F(x_star))

Secant Method - 1 variable
x0 = 2.0, x1 = 3.0
Tolerance = 1.0e-7

iter =  1 | x = 3.0000000000 | F(x) = 7.0000000000e+00
iter =  2 | x = 1.6000000000 | F(x) = 5.6000000000e-01
iter =  3 | x = 1.4782608696 | F(x) = 1.8525519849e-01
iter =  4 | x = 1.4180790960 | F(x) = 1.0948322640e-02
iter =  5 | x = 1.4142990416 | F(x) = 2.4177918928e-04
iter =  6 | x = 1.4142136790 | F(x) = 3.2996208343e-07
iter =  7 | x = 1.4142135624 | F(x) = 9.9715791180e-12

Converged.

Approximate solution: x* = 1.4142135623766205
F(x*) = 9.971579117973306e-12


### Algorithm 8.3: finite difference Newton's method: $n$ variables

![image.png](attachment:1aa7245c-0155-4df3-9bf1-8e79ee2d4971.png)

Example: $F(x)=\left(\begin{array}{c}(x_1+1)^2+ x_2^2 - 2 \\ e^{x_1} + x_2^3 - 2 \end{array}\right)$. Run the example with $x_0= \left(\begin{array}{c} 1 \\ 1 \end{array}\right)$ and $\tau=10^{-7}$

In [5]:
# Algorithm 8.3: Finite difference Newton's method - n variables

function approximate_jacobian(F, x; h=1e-6)
    n = length(x)
    Fx = F(x)
    m = length(Fx)
    J = zeros(m, n)

    for j in 1:n
        e = zeros(n)
        e[j] = 1.0
        J[:, j] = (F(x + h*e) - Fx) / h
    end

    return J
end

function finite_difference_newton_nd(F, x0; τ=1e-7, h=1e-6, max_iter=100)
    x = float.(copy(x0))

    println("Finite Difference Newton Method - n variables")
    println("Initial x = ", x)
    println("Tolerance = ", τ)
    println()

    for k in 0:max_iter
        Fx = F(x)
        normF = norm(Fx)

        println("iter = ", k,
                " | x = ", round.(x, digits=10),
                " | ||F(x)|| = ", normF)

        if normF <= τ
            println("\nConverged.")
            return x
        end

        J = approximate_jacobian(F, x; h=h)

        # Solve J*p = -F(x)
        p = -J \ Fx

        x = x + p
    end

    println("\nMaximum number of iterations reached.")
    return x
end

function F_nd(x)
    return [
        (x[1] + 1)^2 + x[2]^2 - 2,
        exp(x[1]) + x[2]^3 - 2
    ]
end

x0 = [1.0, 1.0]
τ = 1e-7

x_star = finite_difference_newton_nd(F_nd, x0; τ=τ)
println("\nApproximate solution: x* = ", x_star)
println("F(x*) = ", F_nd(x_star))

Finite Difference Newton Method - n variables
Initial x = [1.0, 1.0]
Tolerance = 1.0e-7

iter = 0 | x = [1.0, 1.0] | ||F(x)|| = 3.457237689545305
iter = 1 | x = [0.1523593395, 1.1952816474] | ||F(x)|| = 1.1547093688788046
iter = 2 | x = [-0.0108377082, 1.0361113554] | ||F(x)|| = 0.11404322619252696
iter = 3 | x = [-0.0008897048, 1.0015353733] | ||F(x)|| = 0.003942464318719784
iter = 4 | x = [-1.3718e-6, 1.0000029408] | ||F(x)|| = 8.084656943635373e-6
iter = 5 | x = [-0.0, 1.0] | ||F(x)|| = 3.705397913916308e-11

Converged.

Approximate solution: x* = [-8.437432101158647e-12, 1.0000000000144869]
F(x*) = [1.2098766433155106e-11, 3.502309553482519e-11]


Run the example with $x_0= \left(\begin{array}{c} 1 \\ 1 \end{array}\right)$ and $\tau=0.1$

In [6]:
# Same example with tolerance τ = 0.1

x0 = [1.0, 1.0]
τ = 0.1

x_star = finite_difference_newton_nd(F_nd, x0; τ=τ)
println("\nApproximate solution: x* = ", x_star)
println("F(x*) = ", F_nd(x_star))

Finite Difference Newton Method - n variables
Initial x = [1.0, 1.0]
Tolerance = 0.1

iter = 0 | x = [1.0, 1.0] | ||F(x)|| = 3.457237689545305
iter = 1 | x = [0.1523593395, 1.1952816474] | ||F(x)|| = 1.1547093688788046
iter = 2 | x = [-0.0108377082, 1.0361113554] | ||F(x)|| = 0.11404322619252696
iter = 3 | x = [-0.0008897048, 1.0015353733] | ||F(x)|| = 0.003942464318719784

Converged.

Approximate solution: x* = [-0.0008897048487189491, 1.0015353733350911]
F(x*) = [0.0012944859187404845, 0.0037238865598405724]


### Algorithm 8.4: secant method: $n$ variables

![image.png](attachment:e0ac0e0c-db0e-4acd-b759-628cdff5d12f.png)

Example 7.11: $F(x)=\left(\begin{array}{c}(x_1+1)^2+ x_2^2 - 2 \\ e^{x_1} + x_2^3 - 2 \end{array}\right)$, $x_0=\left(\begin{array}{c} 1 \\ 1 \end{array}\right)$


In [7]:
# Algorithm 8.4: Secant method / Broyden method - n variables
# Example 7.11

function broyden_method(F, x0; τ=1e-7, max_iter=100)
    x = float.(copy(x0))
    n = length(x)

    # Initial approximation of the Jacobian using finite differences
    B = approximate_jacobian(F, x)

    println("Broyden / Secant Method - n variables")
    println("Initial x = ", x)
    println("Tolerance = ", τ)
    println()

    for k in 0:max_iter
        Fx = F(x)
        normF = norm(Fx)

        println("iter = ", k,
                " | x = ", round.(x, digits=10),
                " | ||F(x)|| = ", normF)

        if normF <= τ
            println("\nConverged.")
            return x
        end

        # Solve B*s = -F(x)
        s = -B \ Fx
        x_new = x + s

        F_new = F(x_new)
        y = F_new - Fx

        # Broyden rank-one update:
        # B_{k+1} = B_k + ((y - B_k*s) * s') / (s' * s)
        denom = dot(s, s)
        if denom > 1e-12
            B = B + ((y - B*s) * s') / denom
        else
            println("Step too small. Stopping.")
            return x
        end

        x = x_new
    end

    println("\nMaximum number of iterations reached.")
    return x
end

function F_nd(x)
    return [
        (x[1] + 1)^2 + x[2]^2 - 2,
        exp(x[1]) + x[2]^3 - 2
    ]
end

x0 = [1.0, 1.0]
τ = 1e-7

x_star = broyden_method(F_nd, x0; τ=τ)
println("\nApproximate solution: x* = ", x_star)
println("F(x*) = ", F_nd(x_star))

Broyden / Secant Method - n variables
Initial x = [1.0, 1.0]
Tolerance = 1.0e-7

iter = 0 | x = [1.0, 1.0] | ||F(x)|| = 3.457237689545305
iter = 1 | x = [0.1523593395, 1.1952816474] | ||F(x)|| = 1.1547093688788046
iter = 2 | x = [0.0695579907, 0.9695200131] | ||F(x)|| = 0.0855584607339761
iter = 3 | x = [0.0235690845, 0.9992495342] | ||F(x)|| = 0.05099364569962338
iter = 4 | x = [0.000353738, 1.0025838095] | ||F(x)|| = 0.010030792302490068
iter = 5 | x = [-3.7189e-6, 1.0004069819] | ||F(x)|| = 0.0014606854940871373
iter = 6 | x = [-1.5693e-6, 1.0000015449] | ||F(x)|| = 3.0656652370274317e-6
iter = 7 | x = [-1.83e-8, 1.000000006] | ||F(x)|| = 2.4502020729942282e-8

Converged.

Approximate solution: x* = [-1.826973865625129e-8, 1.0000000060191874]
F(x*) = [-2.4501102036111888e-8, -2.1217649859295307e-10]
